Facebook data downloaded from Humanitarian Data Exchange on 17th January 2025.
Links for 2020 and 2021-2022 data are: https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/55a51014-0d27-49ae-bf92-c82a570c2c6c/download/movement-range-data-2022-05-22.zip
and https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/3d77ce5c-ab6d-4864-b8a2-c8bafffac4f3/download/movement-range-data-2020-03-01-2020-12-31.zip
which are accessible from:
https://data.humdata.org/dataset/movement-range-maps?

In [1]:
#import pandas as pd
import pycountry
import polars as pl
#import re
#import geopandas as gpd
from emu_renewal.inputs import RAW_MOB_PATH, DATA_PATH, get_country_mobility

In [2]:
data20 = pl.read_csv(RAW_MOB_PATH / "movement-range-data-2020-03-01--2020-12-31.txt", separator="\t", schema_overrides={"ds": pl.datatypes.Date})
data21_22 = pl.read_csv(RAW_MOB_PATH / "movement-range-2022-05-22.txt", separator="\t", schema_overrides={"ds": pl.datatypes.Date})
fb_data = pl.concat([data20, data21_22])

In [3]:
country = "NZL"
iso2_code = pycountry.countries.lookup(country).alpha_2
iso3_code = pycountry.countries.lookup(country).alpha_3

# Should be 1 for most European countries, 2 for most others
gadm_level = 2

In [4]:
country_mobility = fb_data.filter(pl.col("country") == iso3_code)

In [5]:
data_col = "all_day_bing_tiles_visited_relative_change"

In [6]:
lookup = [f"GID_{gadm_level}", "polygon_id"]

In [7]:
import json

In [8]:
country_pop_data = json.load(open(DATA_PATH/f"population/gadm_est/{iso3_code}_{gadm_level}.json",'r'))

In [ ]:
country_pop_data

In [ ]:
region_ids = list(country_pop_data.keys())
region_ids

In [11]:
import numpy as np
import pandas as pd

In [ ]:
country_mob_series = pd.Series(index=country_mobility["ds"].unique(),dtype=float)
country_mob_series[:] = 0.0
relevant_pop = 0.0
for pid in region_ids:
    print(pid)
    ##print(pid, pid in country_mobility["polygon_id"].unique())
    if pid in country_mobility["polygon_id"].unique():
        cur_data = country_mobility.filter(pl.col(lookup[1]) == pid)
        mob_series = pd.Series(index=cur_data["ds"].unique(), data = cur_data[data_col])
        mob_series = mob_series.dropna()
        region_pop = float(country_pop_data[pid])
        country_mob_series += (mob_series.reindex(country_mob_series.index,method="nearest")) * region_pop
        relevant_pop += region_pop
weighted_country_mob = country_mob_series / relevant_pop

In [ ]:
weighted_country_mob.plot()

In [14]:
weighted_country_mob.to_csv(DATA_PATH / f"mobility/{iso3_code}_fbmob_data.csv")